In [ ]:
# The demo for the unbranchable paper's infinite family

from modules.branch_dfs import find_z2_merge_structures_dfs, has_z2_merge_structure_dfs
import regina, snappy
from veering.taut import get_taut_structures, isosig_from_tri_angle
from modules.taut_mobius_surgery import get_mobius_strip_indices, unjoin_face, surgery_admissible_faces, unjoin_face, is_surgery_admissible, taut_mobius_dehn_surgery, taut_n_mobius_dehn_surgery


M=snappy.Manifold('m338')
tri=regina.Triangulation3(M)
angle = [2, 2, 0, 0, 2] # angle = [1, 0, 1, 2, 1] the other taut structure on tri. It does not have a face we can use for the example.
face_num = 1
angle_copy = angle[:]
for i in range(10):
    tri_copy = regina.Triangulation3(tri)
    new_t,nangle, nfnum = taut_n_mobius_dehn_surgery(tri_copy,angle_copy,face_num,i)
    M = snappy.Manifold(new_t)
    print(has_z2_merge_structure_dfs(new_t),isosig_from_tri_angle(new_t,nangle), M.identify(), M.volume())

In [ ]:
# the unbranchable subtriangulation is:
tri_copy = regina.Triangulation3(tri)
subtri = unjoin_face(tri_copy,face_num)
assert not has_z2_merge_structure_dfs(subtri) # So, none of the family can have one.

In [ ]:
# Now let's build the surgery parent. This currently not in the module. 

In [ ]:
import itertools

# Copy the parent triangulation.
tri_copy = regina.Triangulation3(tri)
angle_copy = angle[:]


# Pop open along the Möbius-band triangle
face = tri_copy.triangle(face_num)
assert face.isMobiusBand()

embed0 = face.embedding(0)
embed1 = face.embedding(1)

tet0_index, d = embed0.tetrahedron().index(), embed0.face()
tet1_index, s = embed1.tetrahedron().index(), embed1.face()

open_pairs = [ (tet0_index, d), (tet1_index, s)]

# Ungluing one side automatically unglues the other side.
tet, face = open_pairs[0]
tri_copy.tetrahedron(tet).unjoin(face)


# Insert a drilled solid torus. 
dst_sig = "cHcbbpu" # Regina recogniser confirms this is T^2 \times I with one boundary component
dst = regina.Triangulation3.fromIsoSig(dst_sig)
dst.orient()

old_tet_count = tri_copy.countTetrahedra()


# Find the two triangular faces belonging to each boundary
# component of the drilled solid torus.
dst_boundary_pairs = []

for face in [f for f in dst.triangles() if f.isBoundary()]:
    boundary_faces = []
    embed = face.embedding(0)
    dst_boundary_pairs.append((old_tet_count + embed.tetrahedron().index(),embed.face()))

# For this particular DST, each torus boundary should consist
# of two triangles.

# Insert the DST, still unglued.
tri_copy.insertTriangulation(dst)

# Right, now there are many non-canonical ways to insert the solid torus in appropriately.
# Still, the manifold when we find it will be unique. So we brute force our way there.

# Load all possible gluings.
pairing_perms = {}
for gadget_tet, gadget_face in open_pairs:
    for dst_tet, dst_face in dst_boundary_pairs:
        gadget_vertices = [v for v in range(4) if v != gadget_face]
        dst_vertices = [v for v in range(4) if v != dst_face]
        perms = []

        # scan through the perms with correct image
        for perm in itertools.permutations(range(4)):
            p = regina.Perm4(*perm)
        
            if p[gadget_face] == dst_face and p.sign() == -1:
                perms.append(p)
        
        assert len(perms) == 3

        pairing_perms[((gadget_tet, gadget_face), (dst_tet, dst_face))] = perms

candidates = []

# test all gluings, collect candidates. 
for dst_order in itertools.permutations(dst_boundary_pairs):

    perms0 = pairing_perms[(open_pairs[0], dst_order[0])]
    perms1 = pairing_perms[(open_pairs[1], dst_order[1])]

    for p0, p1 in itertools.product(perms0, perms1):

        test = regina.Triangulation3(tri_copy)

        (gadget_tet0, gadget_face0) = open_pairs[0]
        (gadget_tet1, gadget_face1) = open_pairs[1]

        (dst_tet0, dst_face0) = dst_order[0]
        (dst_tet1, dst_face1) = dst_order[1]

        test.tetrahedron(gadget_tet0).join(gadget_face0, test.tetrahedron(dst_tet0), p0)
        test.tetrahedron(gadget_tet1).join(gadget_face1, test.tetrahedron(dst_tet1), p1)

        candidates.append(test)

# Now all manifolds with torus boundary we get should be the same, both mathematically and for our purpose.
assert len(set(c.isoSig() for c in candidates if all(c.isValid() and (v.isStandard() for v in c.vertices()))))==1

# Great. So we can take any one of them.
glued_tri = next(c for c in candidates if all(c.isValid() and (v.isStandard() for v in c.vertices())))

# Now we just have to fill out the angles on the tori. Once again, there is only one way to do this mathematically.
# So we brute force a search (there are only four choices). 
from veering.taut import is_taut

glued_angle = angle_copy[:]
for (i,j) in itertools.product(range(3),repeat=2):
    if is_taut(glued_tri,glued_angle+[i,j]):
        print(isosig_from_tri_angle(glued_tri,glued_angle+[i,j]))

In [ ]:
# Finally, the parent /also/ contains the gadget as a subtriangulation. So:
parent = 'hLvAQkadfdgggfjqxsjnbr_2110202'
tri = regina.Triangulation3('hLvAQkadfdgggfjqxsjnbr')
assert not has_z2_merge_structure_dfs(tri)